In [1]:
import openai
from openai import AsyncOpenAI
from agents.models import openai_provider
from agents import (
set_default_openai_api,
set_default_openai_client,
set_tracing_disabled
)
import os
from agents import Agent,Runner,function_tool

# 对模型进行配置文件的设置，之后可以直接使用

api_key = os.environ["DEEPSEEK_API_KEY"]
url = os.environ["DEEPSEEK_API_BASE_URL"]

client = AsyncOpenAI(api_key= api_key,base_url = url)

# 使用自定义客户端
set_default_openai_client(client = client, use_for_tracing = False)

# 使用兼容的API模式
set_default_openai_api("chat_completions")

# 禁用OpenAI跟踪服务
set_tracing_disabled(disabled = True)

model_name = 'deepseek-chat'

openai_provider.DEFAULT_MODEL = model_name # 自定义模型名称


In [6]:
from agents import function_tool
import sympy
from sympy import symbols, diff, sympify, lambdify
from sympy.parsing.sympy_parser import parse_expr
import re
import json

# 使用function_tool修饰工具
@function_tool
def derivative_calculator(expression: str, variable: str = "x", 
                         evaluate_at: float = None,
                         nth_derivative: int = 1) -> str:
    """
    符号求导计算器
    
    参数:
        expression: 数学表达式字符串，如 "x**2 + sin(x) + 2*x"
        variable: 求导变量，默认为 "x"
        evaluate_at: 在特定点求值，如为None则返回导数表达式
        nth_derivative: 求导的阶数，默认为1（一阶导数）
        
    返回:
        JSON格式的字符串结果
    """
    try:
        # 清理表达式字符串
        expression = expression.strip().replace('^', '**')
        
        # 定义符号变量
        x = symbols(variable)
        
        # 解析表达式
        expr = parse_expr(expression, transformations='all')
        
        # 计算导数
        derivative_expr = diff(expr, x, nth_derivative)
        
        # 简化表达式
        simplified_derivative = sympy.simplify(derivative_expr)
        
        # 准备结果
        result = {
            "original_expression": expression,
            "derivative_expression": str(simplified_derivative),
            "variable": variable,
            "order": nth_derivative,
            "status": "success"
        }
        
        # 如果在特定点求值
        if evaluate_at is not None:
            # 创建lambda函数用于数值计算
            f_derivative = lambdify(x, simplified_derivative, 'numpy')
            try:
                value = float(f_derivative(evaluate_at))
                result["value_at_point"] = value
                result["evaluation_point"] = evaluate_at
            except Exception as e:
                result["evaluation_error"] = f"无法在点 {evaluate_at} 求值: {str(e)}"
        
        return json.dumps(result, ensure_ascii=False)
        
    except Exception as e:
        error_result = {
            "status": "error",
            "error_message": f"计算导数时出错: {str(e)}",
            "original_expression": expression,
            "variable": variable
        }
        return json.dumps(error_result, ensure_ascii=False)

# 使用function_tool修饰工具
@function_tool
def partial_derivative(expression: str, variables: str, 
                      evaluate_at: str = None,
                      order: int = 1) -> str:
    """
    计算偏导数
    
    参数:
        expression: 数学表达式字符串
        variables: 逗号分隔的变量字符串，如 "x,y"
        evaluate_at: JSON字符串格式的求值点，如 '{"x": 1, "y": 2}'
        order: 求导阶数
        
    返回:
        JSON格式的字符串结果
    """
    try:
        # 清理表达式
        expression = expression.strip().replace('^', '**')
        
        # 解析变量字符串
        var_list = [v.strip() for v in variables.split(',')]
        
        # 创建符号变量
        sym_vars = symbols(','.join(var_list))
        
        # 解析表达式
        expr = parse_expr(expression, transformations='all')
        
        # 计算偏导数
        partial_derivatives = {}
        for i, var in enumerate(var_list):
            partial_derivatives[var] = str(diff(expr, sym_vars[i], order))
        
        result = {
            "original_expression": expression,
            "partial_derivatives": partial_derivatives,
            "variables": var_list,
            "order": order,
            "status": "success"
        }
        
        # 如果在特定点求值
        if evaluate_at:
            try:
                eval_dict = json.loads(evaluate_at)
                evaluation_results = {}
                for var, deriv_expr in partial_derivatives.items():
                    try:
                        # 创建求值函数
                        f = lambdify(sym_vars, parse_expr(deriv_expr), 'numpy')
                        # 准备参数
                        args = [eval_dict.get(v, 0) for v in var_list]
                        evaluation_results[var] = float(f(*args))
                    except Exception as e:
                        evaluation_results[var] = f"求值错误: {str(e)}"
                
                result["evaluation_results"] = evaluation_results
                result["evaluation_point"] = eval_dict
            except json.JSONDecodeError as e:
                result["evaluation_error"] = f"解析求值点出错: {str(e)}"
        
        return json.dumps(result, ensure_ascii=False)
        
    except Exception as e:
        error_result = {
            "status": "error",
            "error_message": f"计算偏导数时出错: {str(e)}",
            "original_expression": expression,
            "variables": variables
        }
        return json.dumps(error_result, ensure_ascii=False)

In [7]:
agent = Agent(
    name = "deepseek-chat",
    instructions = "你是我的AI助手，帮助我解决问题" , # 系统提示词
    model = 'deepseek-chat',
    tools=[derivative_calculator, partial_derivative], # 定义工具
)

In [5]:
async def main():
    result = await Runner.run(agent,"计算x^2")
    print(result.final_output)

await main()

我注意到您想要计算x²，但您没有指定要在哪个点计算这个表达式。为了使用求导工具，我需要知道：

1. 您想要计算x²在哪个特定点的值？
2. 或者您其实是想要计算x²的导数？

请告诉我您具体想要做什么：
- 如果是要计算x²在某个点的函数值，请告诉我x的值
- 如果是要计算x²的导数，请告诉我要在哪个点计算导数值

这样我就能帮您准确计算了。
